In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# --- ECG Machine Measurements — Near ED or Hospital Visit (time-varying state) ---
# Machine-generated report text only (no raw waveform measurements).
# Covers:
#   - ED-only patients: ECG within ±1 hr of ED window
#   - Admitted patients: ECG from ED arrival to hospital discharge (or ICU transfer if applicable), ±1 hr
#
# ⚠️  KNOWN LIMITATION: ecg_time is from the ECG machine's internal clock, which is frequently
# out of sync with hospital systems. The ±1 hour buffer is a pragmatic tolerance.
# ~25% of cohort patients are expected to have at least one ECG near their ED visit.

query_ecg = """
WITH cohort_subjects AS (
  SELECT
    ed_stay_id,
    subject_id,
    hadm_id,
    ed_intime,
    -- For admitted patients: cap window at ICU transfer if one occurred, else use dischtime
    CASE
      WHEN hadm_id IS NOT NULL AND first_icu_intime IS NOT NULL
        THEN first_icu_intime
      WHEN hadm_id IS NOT NULL
        THEN dischtime
      ELSE ed_outtime
    END AS window_end
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  cs.ed_stay_id,
  rl.subject_id,
  rl.study_id,
  mm.ecg_time,
  -- Concatenate all 18 machine report text segments
  TRIM(CONCAT(
    COALESCE(mm.report_0,  ''), ' ', COALESCE(mm.report_1,  ''), ' ',
    COALESCE(mm.report_2,  ''), ' ', COALESCE(mm.report_3,  ''), ' ',
    COALESCE(mm.report_4,  ''), ' ', COALESCE(mm.report_5,  ''), ' ',
    COALESCE(mm.report_6,  ''), ' ', COALESCE(mm.report_7,  ''), ' ',
    COALESCE(mm.report_8,  ''), ' ', COALESCE(mm.report_9,  ''), ' ',
    COALESCE(mm.report_10, ''), ' ', COALESCE(mm.report_11, ''), ' ',
    COALESCE(mm.report_12, ''), ' ', COALESCE(mm.report_13, ''), ' ',
    COALESCE(mm.report_14, ''), ' ', COALESCE(mm.report_15, ''), ' ',
    COALESCE(mm.report_16, ''), ' ', COALESCE(mm.report_17, '')
  )) AS machine_report
FROM `physionet-data.mimiciv_ecg.record_list` rl
INNER JOIN `physionet-data.mimiciv_ecg.machine_measurements` mm
  ON rl.study_id = mm.study_id
INNER JOIN cohort_subjects cs
  ON rl.subject_id = cs.subject_id
  AND CAST(mm.ecg_time AS DATETIME) >= DATETIME_SUB(cs.ed_intime, INTERVAL 1 HOUR)
  AND CAST(mm.ecg_time AS DATETIME) <= DATETIME_ADD(cs.window_end, INTERVAL 1 HOUR)
ORDER BY cs.ed_stay_id, mm.ecg_time
"""

print("Running Query 5: ECG machine measurements...")
df_ecg = client.query(query_ecg).to_dataframe()
print(f"Shape: {df_ecg.shape}")

In [ ]:
DESCRIPTION = (
    "ECG machine-generated report text for cohort patients, from mimiciv_ecg.machine_measurements. "
    "Raw waveform measurements excluded; only the concatenated machine report (report_0–report_17) is retained. "
    "Window covers ED arrival through hospital ward stay (capped at ICU transfer if applicable). "
    "±1 hour buffer applied around window boundaries to account for ECG machine clock drift."
)

ds = Dataset.from_pandas(df_ecg, split='ecg')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="ecg", data_dir="ecg")
push_dataset_card("ADS599-Capstone/raw_data", config_name="ecg", split="ecg", data_dir="ecg", description=DESCRIPTION)
print("Pushed ecg data to HuggingFace Hub.")